I used model not from official collab + diffrent methods (more convinient as the ones in colla are a little too old and cause conflicts in dependencies). but model is not instruction based and I followed seminar:)

In [1]:
!pip install transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 374.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 33.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 10.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 44.2 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 31.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 21.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [18]:
!pip install xlstm accelerate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "mistralai/Mistral-7B-v0.1"

# load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")

# text gen
prompt = "Dart vader looked at the skyoker and said: "
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Dart vader looked at the skyoker and said:  “I am your father”

The skyoker looked at him and said: “No way”

Dart vader said: “I am your father”

The skyoker said: “No way”

Dart


In [19]:
def inp_out(input_text, new_tok=50, top_k=250, temperature=1.0):
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=new_tok,
        do_sample=True,
        top_k=top_k,
        temperature=temperature
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

inp_out("Second world war began in ")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'Second world war began in 1939 as Neville Chamberlain resigned and Winston Churchill became Prime Minister.\n\nEvery day, the number of casualties coming home from the war were rising. Britain at the time did not have enough doctors and nurses to'

Task 1 (1 pt): arange a conversation between any two of the following:

a celebrity or politician of your choice
any fictional character (except Darth Vader)
yourself
Compare two setups: a) you prompt with character names only b) you supply additional information (see example).

In [20]:
# Task 1
prompt = '''Adolf Hitler and Elon Musk are having a conversation.
Adolf: '''
print(inp_out(prompt, new_tok=50, temperature=1.0)) #<- character names only

print("--------------------------------------------------")

prompt = '''Adolf Hitler (a nacist that started second worl war) and Elon Musk (a genious inventor and a businessman) are having a conversation about rockets.
Adolf:'''
print(inp_out(prompt, new_tok=100, temperature=1.0))#<-additional infomation

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Adolf Hitler and Elon Musk are having a conversation.
Adolf:  [talking about himself]  “I’ve always wondered that if a cat couldn’t turn the steering wheel well, why weren’t they all dead?”
Elon Musk: “Because I told them through Twitter that we
--------------------------------------------------
Adolf Hitler (a nacist that started second worl war) and Elon Musk (a genious inventor and a businessman) are having a conversation about rockets.
Adolf: Not good enough.
Elon:? What do you mean.
Adolf: Not smart enough!
Elon: I mean, I have a three digit IQ.
Adolf: It begins with two ones and a seven, no?
Elon: (thought for a while) Yes.
Adolf: (laughs)
Elon: I'm older, I should've been afraid of you.
Adolf: (still laughing


Please choose task 2a or 2b (1pt) depending on your model (you can do both, but you will be awarded points for one of these two tasks).

Task 2a: (for BLOOM or other multilingual model) zero-shot translation. Take the first verse of Edgar Allan Poe's "Raven" and translate it into French. (You are free to use any other text of at least the same size)

Original text: ``` Once upon a midnight dreary, while I pondered, weak and weary, Over many a quaint and curious volume of forgotten lore— While I nodded, nearly napping, suddenly there came a tapping, As of some one gently rapping, rapping at my chamber door. “’Tis some visitor,” I muttered, “tapping at my chamber door— Only this and nothing more.”

In [21]:
# Task 2

print("--------------------------------------------------")

english = "Once upon a midnight dreary, while I pondered, weak and weary, Over many a quaint and curious volume of forgotten lore— While I nodded, nearly napping, suddenly there came a tapping, As of some one gently rapping, rapping at my chamber door. “’Tis some visitor,” I muttered, “tapping at my chamber door— Only this and nothing more."


prompt = '''Translatings of the English sentences into French.

- English: What a beautiful day sir, what is your name?
  French: Quelle belle journée, monsieur, comment vous appelez-vous ?
- English: Sharks are very dangerous predators in the ocean
  French: Les requins sont des prédateurs très dangereux dans l'océan
- English:''' + english + '''
  French:'''

output = inp_out(prompt, new_tok=150, top_k=50, temperature=0.2)#<- temperature is low and topp_k is also to make a model translate correctly
# if not set correctly model was outputting on japan language or diffrent sentence in French!
print(output)

print("--------------------------------------------------")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


--------------------------------------------------
Translatings of the English sentences into French.

- English: What a beautiful day sir, what is your name?
  French: Quelle belle journée, monsieur, comment vous appelez-vous ?
- English: Sharks are very dangerous predators in the ocean
  French: Les requins sont des prédateurs très dangereux dans l'océan
- English:Once upon a midnight dreary, while I pondered, weak and weary, Over many a quaint and curious volume of forgotten lore— While I nodded, nearly napping, suddenly there came a tapping, As of some one gently rapping, rapping at my chamber door. “’Tis some visitor,” I muttered, “tapping at my chamber door— Only this and nothing more.
  French: Une fois, dans la nuit d'un milieu de minuit, alors que je me penchais, faible et fatigué, sur de nombreux volumes curieux et oubliés— Alors que je noddais, presque endormi, soudainement, il y avait un tappement, comme de quelqu'un qui tapait doucement, tapant à ma chambre. “C'est un visi

Task 5 -> Beam search

In [24]:
print(inp_out("hi who you are?", new_tok=50, top_k=250, temperature=0.8))
print(inp_out("1+1=2, 2+2=4, ", new_tok=50, top_k=250, temperature=0.8))
# as we can see Mistral-7B is pretrained only not instruction tuned or fine tuned! it continues the statement only

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


hi who you are?

I’m a recent graduate of the University of Pennsylvania’s MFA program in Creative Writing. My work has appeared or is forthcoming in The Arkansas International, Newfound, and the Pittsburgh Poetry Review. I grew up in the
1+1=2, 2+2=4, 3+1=4, 4+2=6, 5+1=6, 6+2=8, 7+1=8, 8+2=10, 9+1=10,


In [145]:
import torch
from typing import Tuple, List

def nucleus_sampling(
    model,
    tokenizer,
    prompt: str,
    prob: float = 0.5
) -> Tuple[str, List[str]]:

    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    logits = logits[0, -1]

    probs = torch.softmax(logits, dim=-1)

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cum_probs = torch.cumsum(sorted_probs, dim=0)

    mask = cum_probs <= prob
    mask[0] = True  # always keep at least one if all below treshold

    nuc_indices = sorted_indices[mask]
    nuc_probs = sorted_probs[mask]

    nuc_probs = nuc_probs / nuc_probs.sum()

    sampled_idx = torch.multinomial(nuc_probs, 1)
    token_id = nuc_indices[sampled_idx].item()

    possible_tokens = [
        tokenizer.convert_ids_to_tokens(i).lstrip("Ġ")
        for i in nuc_indices.tolist()
    ]

    sampled_token = tokenizer.decode(
        [token_id],
        skip_special_tokens=True
    ).strip()

    return sampled_token, possible_tokens


In [146]:
test_prompt = "Elbrus is the highest"
next_token, possible_tokens = nucleus_sampling(model, tokenizer, test_prompt, prob=0.92)
print(test_prompt, next_token, possible_tokens)
assert next_token in possible_tokens
assert 3 <= len(possible_tokens) <= 3
assert sorted(possible_tokens) == ['mountain', 'peak', 'point']

test_prompt = "Large language models can learn to"
next_token, possible_tokens = nucleus_sampling(model, tokenizer, test_prompt, prob=0.4)
print(test_prompt, next_token, possible_tokens)
assert next_token in possible_tokens
# assert sorted(possible_tokens) == ['be', 'communicate', 'do', 'generate', 'perform', 'predict', 'speak', 'write']  <- none sense if I use diffrent model or I should fine tune the prob as in 1 example
# assert len(possible_tokens) == 8

# same mountain as _mountain it is a formatting problem

Elbrus is the highest mountain ['▁mountain', '▁peak', '▁point']


AssertionError: 

In [164]:
import json
import random
import locale; locale.getpreferredencoding = lambda: "UTF-8"
!wget https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json -O aqua.json
data = list(map(json.loads, open("aqua.json")))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


--2025-12-21 16:36:02--  https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 130192 (127K) [text/plain]
Saving to: ‘aqua.json’

aqua.json           100%[===================>] 127.14K   696KB/s    in 0.2s    

2025-12-21 16:36:03 (696 KB/s) - ‘aqua.json’ saved [130192/130192]



In [165]:
print("Example:")
print(data[150])
print(len(data))

Example:
{'question': 'Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?', 'options': ['A)1 minute', 'B)2 minutes', 'C)3 minutes', 'D)4 minutes', 'E)5 minutes'], 'rationale': "Janice's speed = 1/6 miles per minute\nJennie's speed = 1/3 miles per minute\nJanice + Jennie's speed= (1/6 + 1/3) = 1/2 miles per minute\nBoth together will finish the mile in 2 minutes\ncorrect option is B", 'correct': 'B'}
254


In [31]:
EXAMPLE_0SHOT = """
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
""".strip()

In [32]:
batch = tokenizer(EXAMPLE_0SHOT, return_tensors='pt', return_token_type_ids=False).to("cuda")
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_0SHOT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[Prompt:]
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
[Generated:] D
Question: At the beginning of the 2004 season, the San Francisco 49ers were 5-2. They ended the season 6-10. The 49ers 6 wins was what percent of their wins?
A) 100%
B) 80%
C) 60%
D) 40%
E) 20%
Correct Answer: B
Question: The


In [33]:
EXAMPLE_3SHOT_CHAIN_OF_THOUGHT = """
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D


Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Rationale:
""".strip()

In [34]:
device = "cuda"
batch = tokenizer(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
#### NOTE: scroll down for the final answer (below the ======= line)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[Prompt:]
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;
original price = 100*1.6 = 160;
actual price = 160*0.8 = 128.
Answer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125
Number of bags sold = 3000/125 = 24
Answer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If 

Task 6 (1 pt) write a function that automatically creates chain-of-thought prompts. Follow the instructions from the function docstring.

In [35]:
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(*, main_question, fewshot_examples):
  """
  Your goal is to produce the same prompt as the EXAMPLE_3SHOT_CHAIN_OF_THOUGHT automatically

  For each few-shot question, make sure to follow the following rules:
  1. Each question begins with QUESTION_PREFIX, after which you should print the question without leading/traiiling spaces (if any)
  2. After the question, provide space-separated options. Each option should be put in double brackets, followed by option text, e.g. "(A) 146%"
  3. Then, provide the answer as a single letter (A-E)
  4. Finally, add trailing newlines from FEWSHOT_SEPARATOR

  Your final prompt should contain all fewshot_examples (in order), separated with FEWSHOT_SEPARATOR, then follow with main_question.
  The main_question should contain the question and options formatted the same way as in FEWSHOT_EXAMPLES.
  After that, you should prompt the model to produce an explanation (rationale) for the answer.

  Please make sure your prompt contains no leading/trailing newlines or spaces, same as in EXAMPLE_3SHOT_CHAIN_OF_THOUGHT
  """
  parts = []

  def format_example(ex, include_answer=True):
    q = f"{QUESTION_PREFIX}{ex['question'].strip()}"
    opts = " ".join(["("+opt[:2] + " " + opt[2:].strip() for opt in ex['options']])
    s = f"{q}\n{OPTIONS_PREFIX}{opts}"
    if include_answer:
        s += f"\n{CHAIN_OF_THOUGHT_PREFIX}{ex['rationale']}\n{ANSWER_PREFIX}{ex['correct']}"
    return s


  for ex in fewshot_examples:
      parts.append(format_example(ex, include_answer=True))

  parts.append(format_example(main_question, include_answer=False)+'\n'+CHAIN_OF_THOUGHT_PREFIX[:len(CHAIN_OF_THOUGHT_PREFIX)-1])

  return FEWSHOT_SEPARATOR.join(parts)


def find_char_differences(s1, s2):
    max_len = max(len(s1), len(s2))
    differences = []

    for i in range(max_len):
        c1 = s1[i] if i < len(s1) else "<missing>"
        c2 = s2[i] if i < len(s2) else "<missing>"
        if c1 != c2:
            differences.append((i, c1, c2))

    return differences


generated_fewshot_prompt = make_prompt(main_question=data[150], fewshot_examples=(data[30], data[20], data[5]))

string1 = generated_fewshot_prompt
string2 = EXAMPLE_3SHOT_CHAIN_OF_THOUGHT

diffs = find_char_differences(string1, string2)
for idx, char1, char2 in diffs:
    print(f"Index {idx}: '{char1}' vs '{char2}'")

assert generated_fewshot_prompt == EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, "prompts don't match"
assert generated_fewshot_prompt != make_prompt(main_question=data[150], fewshot_examples=())
assert generated_fewshot_prompt.endswith(make_prompt(main_question=data[150], fewshot_examples=()))

print("Well done!")

# Hint: if two prompts do not match, you may find it usefull to use https://www.diffchecker.com or similar to find the difference

Well done!


Task 7 (1 points): Evaluate your prompt.

Please run the model on the entire dataset and measure it's accuracy. For each question, peak  n=5  other questions at random to serve as few-shot examples. Make sure not to accidentally sample the main_question among few-shot examples. For scientific evaluation, it is also a good practice to split the data into two parts: one for eval, and another for few-shot examples. However, doing so is optional in this homework.

The tricky part is when to stop generating: if you don't control for this, your model can accidentally generate a whole new question - and promptyly answer it :) To make sure you get the correct answer, stop generating tokens when the model is done explaining it's solution. To circumvent this, you need to stop generating as soon as the model generates Final Answer: [A-E] To do so, you can either generate manually (see low-level generation above) or use transformers stopping criteria, whichever you prefer.

If you do everything right, the model should be much better than random. However, please do not expect miracles: this is far from the best models, and it will perform much worse than an average human.

In [144]:

# a 2) question, we will try to make one answer wrong and look at performance.
NUM_SAMPLES = 0
NUM_RESPONDED = 0
NUM_CORRECT = 0
FEW_SHOT_N = 3
BATCH_SIZE = 32

import re
import random

# to make one random answer wrong we will look at the output
def get_sequential_few_shots(data, idx, k):
    n = len(data)
    shots = []
    for j in range(k, 0, -1):
        shots.append(data[(idx - j) % n])
    return shots

def generate_until_final_answer_batch(prompts, max_new_tokens=350):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True, 
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
    )

    gen_tokens = outputs[:, prompt_lengths:]

    return tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)

batch_items = []
inc = 0

for idx, sample in enumerate(data):
    NUM_SAMPLES += 1
    if idx % 100 == 0:
        print(f"{idx}/{len(data)}")

    few_shot_examples = get_sequential_few_shots(data, idx, FEW_SHOT_N)
    prompt = make_prompt(main_question=sample, fewshot_examples=few_shot_examples)

    batch_items.append({
        'prompt': prompt,
        'sample': sample,
        'idx': idx
    })

    if len(batch_items) == BATCH_SIZE or idx == len(data) - 1:
        batch_prompts = [item['prompt'] for item in batch_items]
        responses = generate_until_final_answer_batch(batch_prompts)

        for item, response in zip(batch_items, responses):
            s = item['sample']
            global_idx = item['idx']
            prompt = item['prompt']

            if ANSWER_PREFIX in response:
                NUM_RESPONDED += 1
                answer_text = response.split(ANSWER_PREFIX, 1)[1].strip().split("\n", 1)[0]
                match = re.search(r"^[A-E]", answer_text)
                if match:
                    answer_letter = match.group(0).upper()
                    if answer_letter == s["correct"]:
                        NUM_CORRECT += 1
                    else:
                        # print(f"\n❌ Mismatch at idx {global_idx}")
                        # print(f"Model said {answer_letter}, correct is {s['correct']}")
                        # print("Response:\n", response)
                        # print("\nPrompt:\n", prompt)
                        # print("\n" + "="*80 + "\n")
                        inc+=1
                else:
                    print("no match!")
                    print("Response:\n", response)

        batch_items = [] 

print(f"Samples evaluated: {NUM_SAMPLES}")
print(f"Responses produced: {NUM_RESPONDED}")
print(f"Correct answers: {NUM_CORRECT}")
print(f"Accuracy: {NUM_CORRECT / NUM_RESPONDED * 100:.2f}%")
print(inc)



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


0/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


100/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


200/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Samples evaluated: 254
Responses produced: 202
Correct answers: 63
Accuracy: 31.19%
139


In [111]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

Responded %%: 0.9133858267716536
Accuracy (when responded): 0.34913793103448276
Accuracy (overall): 0.3188976377952756


Task 8 (2 points) Experiment time!

Your final quest is to use the testbench you've just written to answer one of the following questions:

Option 1: How many shots do you need?
How does model accuracy change with the number of fewshot examples?

a. check if the model accuracy changes as you increase/decrease the number of "shots"

b. try to prompt-engineer a model into giving the best rationale without any few-shot examples, i.e. zero-shot

For zero-shot mode, feel free to use wild prompt-engineering or modify the inference procedure.

Option 2: Is this prompting tecnique reliable?
Inspired by ongoing research by Anton Voronov, Lena Volf and Max Ryabinin.

For this option, you need to check if the model behavior (and hence, accuracy) is robust to perturbations in the input prompt.

a. Does the accuracy degrade if you provide wrong answers to few-shot examples? (make sure to modify rationale if it contains answer in the end)

b. Does it degrade if you replace question/answer prompts with "Q" and "A"? What if you write both on the same line? Change few-shot separators?

Option 3: Inference Matters
There are many ways to inference the model, not all of them equal.

a. check whether greedy inference or beam search affects model generation quality

b. implement and evaluate sampling with voting (see explanation below).

The voting technique(b) should work as follows: first, you generate k (e.g. 50) "attempts" at an answer using nucleus sampling (or a similar technique). Then, you count how many of those attempts chose a particular option (A, B, etc) as the final answer. The option that was chosen most frequently has the most "votes", and therefore "wins".

To speed up voting, you may want to generate these attempts in parallel as a batch. That should be very easy to implement: just run model.generate on a list with multiple copies of the same prompt.

================================================

Common rules: You will need to test both hypothes (A and B) in the chosen option. You may choose to replace one of them with your own idea - but please ask course staff in advance (via telegram) if you want full points.

Feel free to organize your code and report as you see fit - but please make sure it's readable and the code runs top-to-bottom :) Write a short informal report about what you tried and, in doing so, what did you found. Minimum of 2 paragraphs; more is ok; creative visualizations are welcome.

You are allowed (but not required) to prompt the model into generating a report for you --- or helping you write one. However, if you do so, make sure that it is still human-readable :)

In [143]:
# answer to the question a) 1
## num correct vs num examples ## I will count correct/responded !
#  ~32% - 2
#  ~37%- 3 <- the best is 3 examples for this Mistral-7B
# ~35% - 4
#  ~ 35.5% - 5

## but results are awfull :(, but the model is 7B not 13B as in official turorial

# more is diminishing returns!

# answer to the question b) 1
NUM_SAMPLES = 0
NUM_RESPONDED = 0
NUM_CORRECT = 0
BATCH_SIZE = 32

PROMPT_EX = (
    "You are a highly skilled student solving a multiple-choice question.\n"
    "Each question is independent.\n"
    "Output only the final answer, without explanation.\n"
    "Respond with exactly one capital letter: A, B, C, D, E.\n\n"
)

def make_zero_shot_prompt(sample):
    return (
        PROMPT_EX +
        f"Question:\n{sample['question']}\n\n"
        f"A) {sample['options'][0]}\n"
        f"B) {sample['options'][1]}\n"
        f"C) {sample['options'][2]}\n"
        f"D) {sample['options'][3]}\n"
        f"E) {sample['options'][4]}\n\n"
        "Correct Answer:"
    )


def generate_until_final_answer_batch(prompts, max_new_tokens=5):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,   
        temperature=0.0,   
    )

    gen_tokens = outputs[:, prompt_lengths:]
    return tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)

batch_items = []
inc = 0

for idx, sample in enumerate(data):
    NUM_SAMPLES += 1
    if idx % 100 == 0:
        print(f"{idx}/{len(data)}")

    prompt = make_zero_shot_prompt(sample)

    batch_items.append({
        'prompt': prompt,
        'sample': sample,
        'idx': idx
    })

    if len(batch_items) == BATCH_SIZE or idx == len(data) - 1:
        batch_prompts = [item['prompt'] for item in batch_items]
        responses = generate_until_final_answer_batch(batch_prompts)

        for item, response in zip(batch_items, responses):
            s = item['sample']

            match = re.search(r"[A-E]", response)
            if match:
                NUM_RESPONDED += 1
                if match.group(0) == s["correct"]:
                    NUM_CORRECT += 1
                # else:
                    # print("->",response)

        batch_items = []


print(f"Samples evaluated: {NUM_SAMPLES}")
print(f"Responses produced: {NUM_RESPONDED}")
print(f"Correct answers: {NUM_CORRECT}")
print(f"Accuracy: {NUM_CORRECT / NUM_SAMPLES * 100:.2f}%")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


0/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


100/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


200/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Samples evaluated: 254
Responses produced: 241
Correct answers: 32
Accuracy: 12.60%


In [128]:
prompt_beg = "A person is asking a question: "
prompt_end = "A highly intelligent assistant asnwering breifly to the given question: "
print(inp_out(prompt_beg + "what is a capital of France?" + prompt_end, new_tok=50, top_k=250, temperature=0.8))
# it is working!

print("second ------------------------------>")

print(inp_out(prompt_beg + "what will be 50*12?" + "Very smart mathimatician answering: ", new_tok=50, top_k=350, temperature=1.0))
#correct!

print("third --------------------------------->")
### will try out question answer
print(inp_out( "Question: when do second world war began? Answer:" , new_tok=50, top_k=350, temperature=1.0))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


A person is asking a question: what is a capital of France?A highly intelligent assistant asnwering breifly to the given question:  Paris

A picture of Paris

Even more information about it which the assistant is proivding:

Paris, in the Île-de-France region, is the capital and most populous city of France. It
second ------------------------------>


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


A person is asking a question: what will be 50*12?Very smart mathimatician answering: 130 You ask again: wrong - it is 600.He will kill yo, if you continue ..

A mathematician and a lawyer are sitting next to each other on a long flight. The lawyer kicks
third --------------------------------->
Question: when do second world war began? Answer: on the 1st of September, 1939, with the german attack on poland.

For this timeline the ww2 starts already on the 23.08.1939 – when


In [167]:
# option a 2) Does the accuracy degrade if you provide wrong answers to few-shot examples? (make sure to modify rationale if it contains answer in the end)
NUM_SAMPLES = 0
NUM_RESPONDED = 0
NUM_CORRECT = 0
FEW_SHOT_N = 3 # recomended max 37% we will look if I provide wrong answers and outout degradates.
BATCH_SIZE = 32

print(data[150]) # B

import random
import copy

def get_sequential_few_shots(data, idx, k):
    rd = random.randint(1, k) 
    n = len(data)
    shots = []

    for j in range(k, 0, -1):
        original = data[(idx - j) % n]
        sample = copy.deepcopy(original) # to not overwrite!

        if sample['correct'] == "A":
            sample['correct'] = "B"
        else:
            sample['correct'] = "A"

        shots.append(sample)

    return shots

def generate_until_final_answer_batch(prompts, max_new_tokens=250):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True, 
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
    )

    gen_tokens = outputs[:, prompt_lengths:]

    return tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)

batch_items = []
inc = 0

for idx, sample in enumerate(data):
    NUM_SAMPLES += 1
    if idx % 100 == 0:
        print(f"{idx}/{len(data)}")

    few_shot_examples = get_sequential_few_shots(data, idx, FEW_SHOT_N)
    # prompt = PROMPT_EX + make_prompt(main_question=sample, fewshot_examples=few_shot_examples)
    prompt = make_prompt(main_question=sample, fewshot_examples=few_shot_examples)

    batch_items.append({
        'prompt': prompt,
        'sample': sample,
        'idx': idx
    })

    if len(batch_items) == BATCH_SIZE or idx == len(data) - 1:
        batch_prompts = [item['prompt'] for item in batch_items]
        responses = generate_until_final_answer_batch(batch_prompts)

        for item, response in zip(batch_items, responses):
            s = item['sample']
            global_idx = item['idx']
            prompt = item['prompt']

            if ANSWER_PREFIX in response:
                NUM_RESPONDED += 1
                answer_text = response.split(ANSWER_PREFIX, 1)[1].strip().split("\n", 1)[0]
                match = re.search(r"^[A-E]", answer_text)
                if match:
                    answer_letter = match.group(0).upper()
                    if answer_letter == s["correct"]:
                        NUM_CORRECT += 1
                    else:
                        # print(f"\n❌ Mismatch at idx {global_idx}")
                        # print(f"Model said {answer_letter}, correct is {s['correct']}")
                        # print("Response:\n", response)
                        # print("\nPrompt:\n", prompt)
                        # print("\n" + "="*80 + "\n")
                        inc+=1
                else:
                    print("no match!")
                    print("Response:\n", response)

        batch_items = [] 

print(f"Samples evaluated: {NUM_SAMPLES}")
print(f"Responses produced: {NUM_RESPONDED}")
print(f"Correct answers: {NUM_CORRECT}")
print(f"Accuracy: {NUM_CORRECT / NUM_RESPONDED * 100:.2f}%")
print(inc)

## as we can see queslity degradated: from 37% -> 34% (-2%) (only 1 wrong out of 3), what if I make all answers wrong: 37% -> 24% (-13%)
# I think it is as when you type misspelling it also degradates, the same goes with answers, maybe this gradient descent assumes that it should answer wrong

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{'question': 'Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?', 'options': ['A)1 minute', 'B)2 minutes', 'C)3 minutes', 'D)4 minutes', 'E)5 minutes'], 'rationale': "Janice's speed = 1/6 miles per minute\nJennie's speed = 1/3 miles per minute\nJanice + Jennie's speed= (1/6 + 1/3) = 1/2 miles per minute\nBoth together will finish the mile in 2 minutes\ncorrect option is B", 'correct': 'B'}
0/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


no match!
Response:
 45 + (-30) = 15
Final answer:


Question: A certain company has 100 employees. If 10 of the employees are women, what percent of the employees are women?
Answer Choices: (A) 10% (B) 15% (C) 20% (D) 25% (E) 30%
Rationale: 10/100 = 10%
Final answer:


Question: A certain company has 100 employees. If 10 of the employees are women, what percent of the employees are women?
Answer Choices: (A) 10% (B) 15% (C) 20% (D) 25% (E) 30%
Rationale: 10/100 = 10%
Final answer:


Question: A certain company has 100 employees. If 10 of the employees are women, what percent of the employees are women?
Answer Choices: (A) 10%


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


100/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


200/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Samples evaluated: 254
Responses produced: 204
Correct answers: 50
Accuracy: 24.51%
153


In [ ]:
# b 2) Does it degrade if you replace question/answer prompts with "Q" and "A"? What if you write both on the same line? Change few-shot separators?
# now instead of Question, Answer Response ... we will place Q, A and no Reasoning.

def make_prompt(*, main_question, fewshot_examples):
  parts = []

  def format_example(ex, include_answer=True):
    q = f"Q: {ex['question'].strip()}"
    opts = " ".join(["("+opt[:2] + " " + opt[2:].strip() for opt in ex['options']])
    s = f"{q}\n{OPTIONS_PREFIX}{opts}"
    if include_answer:
        s += f"\nA: {ex['correct']}"
    return s


  for ex in fewshot_examples:
      parts.append(format_example(ex, include_answer=True))

  parts.append(format_example(main_question, include_answer=False)+'\n')

  return FEWSHOT_SEPARATOR.join(parts)


NUM_SAMPLES = 0
NUM_RESPONDED = 0
NUM_CORRECT = 0
FEW_SHOT_N = 3 # recomended max 37% we will look if I provide wrong answers and outout degradates.
BATCH_SIZE = 32

def get_sequential_few_shots(data, idx, k):
    n = len(data)
    shots = []
    for j in range(k, 0, -1):
        sample = data[(idx - j) % n]
        shots.append(sample)
    return shots

def generate_until_final_answer_batch(prompts, max_new_tokens=350):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True, 
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
    )

    gen_tokens = outputs[:, prompt_lengths:]

    return tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)

batch_items = []
inc = 0

for idx, sample in enumerate(data):
    NUM_SAMPLES += 1
    if idx % 100 == 0:
        print(f"{idx}/{len(data)}")

    few_shot_examples = get_sequential_few_shots(data, idx, FEW_SHOT_N)
    prompt = make_prompt(main_question=sample, fewshot_examples=few_shot_examples)

    batch_items.append({
        'prompt': prompt,
        'sample': sample,
        'idx': idx
    })

    if len(batch_items) == BATCH_SIZE or idx == len(data) - 1:
        batch_prompts = [item['prompt'] for item in batch_items]
        responses = generate_until_final_answer_batch(batch_prompts)

        for item, response in zip(batch_items, responses):
            s = item['sample']
            global_idx = item['idx']
            prompt = item['prompt']

            if "A: " in response:
                NUM_RESPONDED += 1
                answer_text = response.split("A: ", 1)[1].strip().split("\n", 1)[0]
                match = re.search(r"^[A-E]", answer_text)
                if match:
                    answer_letter = match.group(0).upper()
                    if answer_letter == s["correct"]:
                        NUM_CORRECT += 1
                    else:
                        # print(f"\n❌ Mismatch at idx {global_idx}")
                        # print(f"Model said {answer_letter}, correct is {s['correct']}")
                        # print("Response:\n", response)
                        # print("\nPrompt:\n", prompt)
                        # print("\n" + "="*80 + "\n")
                        inc+=1
                else:
                    print("no match!")
                    print("Response:\n", response)

        batch_items = [] 

print(f"Samples evaluated: {NUM_SAMPLES}")
print(f"Responses produced: {NUM_RESPONDED}")
print(f"Correct answers: {NUM_CORRECT}")
print(f"Accuracy: {NUM_CORRECT / NUM_RESPONDED * 100:.2f}%")
print(inc)

## so the results are: 37% -> 27.5% which means it is better to give full Answer or Question rather than A, Q
# I thunk it is because "Answer" is a full token in models vocab while "A" is only a letter and in latent space they represent diffrent things

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


0/254


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [180]:
def inspect_tokens(text, tokenizer):
    token_ids = tokenizer.encode(text, add_special_tokens=False)
 
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    
    print(f"Original text:\n{text}\n")
    print(f"Number of tokens: {len(tokens)}\n")
    
    print("Token IDs and corresponding tokens:")
    for tid, tok in zip(token_ids, tokens):
        print(f"{tid:>5} : {tok}")
    
    return token_ids, tokens

print(inspect_tokens(ANSWER_PREFIX, tokenizer))
print(inspect_tokens("A: ", tokenizer))

# as you can see it has ▁answer as a whole token with all its meaning while _A is a just letter

def get_token_embedding(tokenizer, model, token_id):
    embedding_matrix = model.get_input_embeddings().weight 
    embedding_vector = embedding_matrix[token_id].detach()
    return embedding_vector

token_ids = [tokenizer.encode(t, add_special_tokens=False)[0] for t in ["answer", "A"]]

emb1 = get_token_embedding(tokenizer, model, token_ids[0])
emb2 = get_token_embedding(tokenizer, model, token_ids[1])

print("Embedding shape:", emb1.shape)

# cosine similarity
cos_sim = torch.nn.functional.cosine_similarity(emb1, emb2, dim=0)
print("Cosine similarity:", cos_sim.item())
print("---less than a 2%------")

Original text:
Final answer:

Number of tokens: 3

Token IDs and corresponding tokens:
10222 : ▁Final
 4372 : ▁answer
28747 : :
([10222, 4372, 28747], ['▁Final', '▁answer', ':'])
Original text:
A: 

Number of tokens: 3

Token IDs and corresponding tokens:
  330 : ▁A
28747 : :
28705 : ▁
([330, 28747, 28705], ['▁A', ':', '▁'])
Embedding shape: torch.Size([4096])
Cosine similarity: 0.0228118896484375
---less than a 2%------
